# 📖 Notebook 2: Read Receipts & Presence

Welcome back! In the previous notebook we learned how messages are sent, stored,
and delivered. Now we'll explore two features that make messaging apps feel
**alive** — read receipts and presence.

## 🎯 Learning Objectives

By the end of this notebook you will understand:

1. **The checkmark system** — how ✓ (sent), ✓✓ (delivered), and 🔵✓✓ (read) work
2. **Presence tracking** — how the server knows if a user is online or offline
3. **Heartbeat mechanism** — how the client keeps its "online" status alive
4. **Scaling challenges** — what happens when millions of users need real-time status updates

---

```
  This Notebook
  ─────────────
  📓 Notebook 1: Message Delivery & Storage   ✅ done
  📓 Notebook 2: Read Receipts & Presence      ◀── YOU ARE HERE
  📓 Notebook 3: Group Messaging
  📓 Notebook 4: End-to-End Encryption Basics
```

## 🐳 Setup — Make Sure the Infrastructure Is Running

Before running any code, start the services:

```bash
cd system-designs/whatsapp
docker compose up -d
```

This launches:

| Service        | Address             | Purpose                       |
|----------------|---------------------|-------------------------------|
| PostgreSQL     | `localhost:5432`    | Message & inbox storage       |
| Redis          | `localhost:6379`    | Presence, pub/sub             |
| Chat Server    | `ws://localhost:8765` | WebSocket messaging server  |
| Adminer        | `http://localhost:8080` | DB viewer (optional)      |
| RedisInsight   | `http://localhost:5540` | Redis viewer (optional)   |

### 🐍 Kernel Selection

Make sure you select the **`.venv`** kernel in VS Code (top-right kernel picker).
If you haven't created it yet:

```bash
cd system-designs/whatsapp
uv venv
source .venv/bin/activate
uv sync
```

> 💡 If the kernel doesn't appear, reload VS Code (`Cmd+Shift+P` → "Reload Window").

## 🔌 Connection Setup

Let's import our libraries and set up connections to PostgreSQL, Redis, and the
WebSocket server.

In [ ]:
import json
import time

import psycopg2
import psycopg2.extras
import redis
from websockets.sync.client import connect as ws_connect

# --- Database config (same as docker-compose) ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "whatsapp_demo",
    "user": "demo",
    "password": "demo",
}

WS_URL = "ws://localhost:8765"


# --- Helper: run a SQL query and return rows as dicts ---
def query(sql, params=None):
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql, params)
            try:
                return cur.fetchall()
            except psycopg2.ProgrammingError:
                return []
    finally:
        conn.close()


def execute(sql, params=None):
    """Run a SQL statement that modifies data (INSERT/UPDATE/DELETE)."""
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql, params)
            conn.commit()
            try:
                return cur.fetchall()
            except psycopg2.ProgrammingError:
                return []
    finally:
        conn.close()


# --- Helper: print rows as a nice table ---
def print_rows(rows, title=""):
    if title:
        print(f"\n📋 {title}")
        print("─" * 60)
    if not rows:
        print("  (no rows)")
        return
    headers = list(rows[0].keys())
    print("  " + " | ".join(f"{h:>15}" for h in headers))
    print("  " + "-+-".join("-" * 15 for _ in headers))
    for row in rows:
        print("  " + " | ".join(f"{str(row[h]):>15}" for h in headers))


# --- Redis connection ---
r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# --- Quick connection test ---
try:
    users = query("SELECT id, username FROM users ORDER BY id")
    print("✅ PostgreSQL connected! Users in the system:")
    for u in users:
        print(f"   👤 {u['username']} (id={u['id']})")
except Exception as e:
    print(f"❌ PostgreSQL error: {e}")

try:
    r.ping()
    print("\n✅ Redis connected!")
except Exception as e:
    print(f"❌ Redis error: {e}")

try:
    ws_test = ws_connect(WS_URL)
    ws_test.close()
    print("✅ WebSocket server reachable!")
except Exception as e:
    print(f"❌ WebSocket error: {e}")

---

# ✅ Section 1: The Checkmark System

You've seen these in WhatsApp. Every message shows its delivery status:

| Symbol | Meaning | When it happens |
|--------|---------|-----------------|
| ✓      | **Sent** — the server stored your message | Server ACKs the client |
| ✓✓     | **Delivered** — the recipient's device downloaded it | Recipient sends `ack` |
| 🔵✓✓   | **Read** — the recipient opened the chat | Recipient sends `read` |

### How does this work under the hood?

```
  Alice's Phone              Server               Bob's Phone
  ────────────              ────────              ───────────
       │                       │                       │
       │── send_message ──────▶│                       │
       │                       │── store in DB         │
       │◀── ack (stored) ──────│   (inbox: pending)    │
       │   ✓ sent              │                       │
       │                       │── new_message ───────▶│
       │                       │                       │── ack ──────▶│
       │                       │   (inbox: delivered)  │
       │   ✓✓ delivered        │                       │
       │                       │                       │
       │                       │              Bob opens chat
       │                       │                       │── read ─────▶│
       │                       │   (inbox: read)       │
       │◀── read_receipt ──────│                       │
       │   🔵✓✓ read           │                       │
```

### The `inbox` table tracks this

Every time a message is sent, the server creates an **inbox row** for each
recipient. The `status` column transitions through these states:

```
  ┌─────────┐      ack       ┌───────────┐      read      ┌──────┐
  │ pending │ ──────────────▶ │ delivered │ ──────────────▶ │ read │
  └─────────┘                 └───────────┘                 └──────┘
```

Let's see this in action!

---

## 👁️ Section 2: Watching Delivery Status

Let's walk through the entire lifecycle of a message — from sent to delivered —
and watch the `inbox` table change in real time.

In [ ]:
# ── Step 1: Look at the current inbox ──
# The seed data already has some pending messages.

rows = query("""
    SELECT i.id, u.username AS recipient, m.content,
           i.status, i.delivered_at, i.read_at
    FROM inbox i
    JOIN users u ON i.user_id = u.id
    JOIN messages m ON i.message_id = m.id
    ORDER BY i.id
""")
print_rows(rows, "Current Inbox (before our experiment)")

In [ ]:
# ── Step 2: Alice sends a new message to Bob via WebSocket ──
#
# Alice = user 1, Bob = user 2, their 1:1 chat = chat 1

alice_ws = ws_connect(WS_URL)

# First message must be "connect"
alice_ws.send(json.dumps({"type": "connect", "user_id": 1}))
resp = json.loads(alice_ws.recv())
print(f"🔗 Alice connected: {resp}")

# Now send a message
alice_ws.send(json.dumps({
    "type": "send_message",
    "chat_id": 1,
    "content": "Hey Bob, did you finish the homework?"
}))
ack = json.loads(alice_ws.recv())
print(f"\n✓ Server acknowledged (SENT): {ack}")
new_msg_id = ack["message_id"]
print(f"  → Message ID: {new_msg_id}")
print(f"  → At this point Alice sees: ✓ (one grey checkmark)")

In [ ]:
# ── Step 3: Check the inbox — there should be a new 'pending' row for Bob ──

time.sleep(0.5)  # give the server a moment

rows = query("""
    SELECT i.id, u.username AS recipient, m.content,
           i.status, i.delivered_at, i.read_at
    FROM inbox i
    JOIN users u ON i.user_id = u.id
    JOIN messages m ON i.message_id = m.id
    WHERE i.message_id = %s
""", (new_msg_id,))

print_rows(rows, f"Inbox for message {new_msg_id} — status should be 'pending'")
print("\n💡 The message is stored, but Bob hasn't received it yet.")
print("   inbox status = 'pending' → Alice sees ✓ (sent)")

In [ ]:
# ── Step 4: Bob connects and receives the message ──

bob_ws = ws_connect(WS_URL)
bob_ws.send(json.dumps({"type": "connect", "user_id": 2}))
resp = json.loads(bob_ws.recv())
print(f"🔗 Bob connected: {resp}")

# Give Redis pub/sub a moment to deliver
time.sleep(1)

# Bob should have received the new message via Redis pub/sub
# (The server publishes to user:2 channel when Alice sends)
# Let's check by doing a sync to get any pending messages
bob_ws.send(json.dumps({"type": "sync"}))

# Read all sync messages
while True:
    msg = json.loads(bob_ws.recv())
    if msg["type"] == "sync_complete":
        print(f"\n📬 Sync complete — {msg['count']} pending message(s) delivered")
        break
    print(f"📩 Bob received: {msg.get('content', msg)}")

In [ ]:
# ── Step 5: Bob ACKs the message → status changes to 'delivered' ──
#
# In a real app, the client sends an ACK automatically when it
# receives and stores a message locally.

bob_ws.send(json.dumps({"type": "ack", "message_id": new_msg_id}))
ack_resp = json.loads(bob_ws.recv())
print(f"✓✓ Bob acknowledged delivery: {ack_resp}")

time.sleep(0.5)

# Check inbox again
rows = query("""
    SELECT i.id, u.username AS recipient, i.status,
           i.delivered_at, i.read_at
    FROM inbox i
    JOIN users u ON i.user_id = u.id
    WHERE i.message_id = %s
""", (new_msg_id,))

print_rows(rows, f"Inbox for message {new_msg_id} — now 'delivered'!")
print("\n💡 Notice:")
print("   • status changed: pending → delivered")
print("   • delivered_at now has a timestamp")
print("   • Alice would now see ✓✓ (two grey checkmarks)")

---

## 📖 Section 3: Read Receipts

Now let's complete the lifecycle. Bob opens the chat and reads the message.
When that happens:

1. Bob's app sends `{"type": "read", "message_id": N}`
2. Server updates `inbox.status` to `'read'` and sets `read_at`
3. Server publishes a **read_receipt** event to Alice via Redis pub/sub
4. Alice sees 🔵✓✓ (blue checkmarks)

```
  Bob's Phone                Server                Alice's Phone
  ───────────               ────────               ─────────────
       │                       │                         │
       │── read (msg_id) ─────▶│                         │
       │                       │── UPDATE inbox          │
       │                       │   status='read'         │
       │                       │   read_at=NOW()         │
       │                       │                         │
       │                       │── Redis PUBLISH ────────│
       │                       │   user:{alice_id}       │
       │                       │   {read_receipt}        │
       │                       │                    ────▶│
       │                       │               🔵✓✓ shown│
       │◀── read_ok ──────────│                         │
```

In [ ]:
# ── Step 6: Bob reads the message ──

bob_ws.send(json.dumps({"type": "read", "message_id": new_msg_id}))
read_resp = json.loads(bob_ws.recv())
print(f"🔵✓✓ Bob marked as read: {read_resp}")

time.sleep(0.5)

# Check the inbox one more time
rows = query("""
    SELECT i.id, u.username AS recipient, i.status,
           i.delivered_at, i.read_at
    FROM inbox i
    JOIN users u ON i.user_id = u.id
    WHERE i.message_id = %s
""", (new_msg_id,))

print_rows(rows, f"Inbox for message {new_msg_id} — now 'read'!")
print("\n💡 The full lifecycle is complete:")
print("   pending  → ✓  sent")
print("   delivered → ✓✓  delivered")
print("   read     → 🔵✓✓  read")

In [ ]:
# ── Step 7: Alice receives the read_receipt via her WebSocket ──
#
# When Bob sent "read", the server published a read_receipt event
# to Redis channel user:1 (Alice). Alice's WebSocket connection
# receives it automatically through the server's Redis subscriber.

try:
    # Alice should have a read_receipt message waiting
    receipt = json.loads(alice_ws.recv(timeout=3))
    print(f"📨 Alice received notification: {json.dumps(receipt, indent=2)}")
    print(f"\n💡 Alice now knows:")
    print(f"   • Message {receipt.get('message_id')} was read")
    print(f"   • Reader: user {receipt.get('reader_id')} (Bob)")
    print(f"   • Chat: {receipt.get('chat_id')}")
    print(f"   → Alice's app updates the checkmarks to 🔵✓✓")
except Exception as e:
    print(f"⏰ No notification received (timeout): {e}")
    print("   This can happen if the read_receipt was already consumed.")

In [ ]:
# ── Let's also verify via Redis pub/sub directly ──
#
# We can use Redis pub/sub to see the read_receipt event.
# This is what the server does internally.

print("🔍 How read receipts flow through Redis pub/sub:\n")
print("   1. Bob sends:  {type: 'read', message_id: N}")
print("   2. Server:     UPDATE inbox SET status='read', read_at=NOW()")
print("   3. Server:     Looks up who sent the original message (Alice)")
print("   4. Server:     redis.publish('user:1', {type: 'read_receipt', ...})")
print("   5. Subscriber: Listens on 'user:1' channel, forwards to Alice's WS")
print("   6. Alice's app: Updates UI to show 🔵✓✓")

# Show what channels exist
channels = r.pubsub_channels("user:*")
print(f"\n📡 Active Redis pub/sub channels: {channels}")
print("   Each connected user has their own channel.")

In [ ]:
# Clean up WebSocket connections from the delivery/read demo
alice_ws.close()
bob_ws.close()
time.sleep(0.5)
print("🧹 Closed Alice and Bob's WebSocket connections.")

---

# 🟢 Section 4: Presence Tracking

How does WhatsApp know if someone is "online" or show "last seen 5 min ago"?

The answer: **Redis keys with TTL (Time To Live)**.

### How It Works

```
  Redis Memory
  ────────────────────────────────────────────
  KEY                VALUE        TTL
  ────────────────────────────────────────────
  presence:1         "online"     58s remaining
  presence:2         "online"     42s remaining
  presence:3         (expired — key deleted!)
  last_seen:3        "2024-01-15T10:30:00"   (no TTL)
  ────────────────────────────────────────────
```

### The Rules

1. **When a user connects:** set `presence:{user_id}` = `"online"` with a **60-second TTL**
2. **Every heartbeat:** refresh the TTL back to 60 seconds
3. **When the user disconnects:** delete `presence:{user_id}` and set `last_seen:{user_id}`
4. **If the client crashes:** no heartbeat → TTL expires → key auto-deleted → user appears offline

### Why Use TTL Instead of Just Deleting the Key?

```
  Normal disconnect:      Client crashes:
  ──────────────────      ─────────────────
  Client says "bye"       Client vanishes
  Server deletes key      ... silence ...
  ✅ instant offline      60s later: TTL expires
                          ✅ auto offline
```

The TTL is a **safety net**. If a client's internet drops, the server never
gets a disconnect event. Without TTL, that user would appear online forever!

> 💡 **Real-world analogy:** It's like a parking meter. You keep feeding it
> coins (heartbeats) to stay parked (online). If you stop paying, the meter
> expires and you get a ticket (marked offline).

## 🔬 Section 5: Presence in Action

Let's manipulate presence directly in Redis and through the WebSocket server.

In [ ]:
# ── Direct Redis presence ──
#
# First let's clean any leftover presence keys and check the state.

# Clear old presence data for a clean demo
for uid in range(1, 6):
    r.delete(f"presence:{uid}")
    r.delete(f"last_seen:{uid}")

print("🔍 Checking presence for all users (before connecting):")
print("─" * 50)
for uid in range(1, 6):
    username = query("SELECT username FROM users WHERE id = %s", (uid,))[0]["username"]
    online = r.get(f"presence:{uid}")
    last_seen = r.get(f"last_seen:{uid}")
    if online:
        print(f"  🟢 {username}: ONLINE")
    elif last_seen:
        print(f"  ⚪ {username}: offline (last seen {last_seen})")
    else:
        print(f"  ⚫ {username}: offline (never connected)")

In [ ]:
# ── Connect Alice via WebSocket → she becomes online ──

alice_ws = ws_connect(WS_URL)
alice_ws.send(json.dumps({"type": "connect", "user_id": 1}))
resp = json.loads(alice_ws.recv())
print(f"🔗 Alice connected: {resp}")

time.sleep(0.5)

# Check Redis directly
online_val = r.get("presence:1")
ttl_val = r.ttl("presence:1")
print(f"\n🔍 Redis key 'presence:1':")
print(f"   Value: {online_val}")
print(f"   TTL:   {ttl_val} seconds")
print(f"\n🟢 Alice is ONLINE! The key will auto-expire in ~{ttl_val}s without heartbeats.")

In [ ]:
# ── Use the WebSocket get_presence command ──
#
# Any connected client can ask about another user's presence.

# Connect Bob so he can ask about Alice
bob_ws = ws_connect(WS_URL)
bob_ws.send(json.dumps({"type": "connect", "user_id": 2}))
json.loads(bob_ws.recv())  # connected ack

# Bob checks: is Alice online?
bob_ws.send(json.dumps({"type": "get_presence", "user_id": 1}))
presence = json.loads(bob_ws.recv())
print(f"👀 Bob asks: Is Alice online?")
print(f"   Server says: {json.dumps(presence, indent=2)}")

# Bob checks: is Charlie online? (Charlie never connected)
bob_ws.send(json.dumps({"type": "get_presence", "user_id": 3}))
presence = json.loads(bob_ws.recv())
print(f"\n👀 Bob asks: Is Charlie online?")
print(f"   Server says: {json.dumps(presence, indent=2)}")

In [ ]:
# ── Disconnect Alice → she becomes offline with a last_seen timestamp ──

alice_ws.close()
time.sleep(1)  # give the server a moment to process the disconnect

# Check Redis directly
online_val = r.get("presence:1")
last_seen = r.get("last_seen:1")
print(f"🔍 After Alice disconnects:")
print(f"   presence:1  = {online_val}  (key was deleted)")
print(f"   last_seen:1 = {last_seen}")

# Bob checks Alice's presence again
bob_ws.send(json.dumps({"type": "get_presence", "user_id": 1}))
presence = json.loads(bob_ws.recv())
print(f"\n👀 Bob asks again: Is Alice online?")
print(f"   Server says: {json.dumps(presence, indent=2)}")
print(f"\n⚪ Alice is OFFLINE. Bob sees \"last seen {last_seen}\"")

In [ ]:
# Clean up Bob's connection
bob_ws.close()
time.sleep(0.5)
print("🧹 Closed Bob's WebSocket connection.")

---

# 💓 Section 6: The Heartbeat Mechanism

The heartbeat is a simple "I'm still here!" ping sent by the client.

### Why Do We Need Heartbeats?

Without heartbeats, a user who crashes or loses internet would appear
online **forever** — the server never gets a clean disconnect event.

```
  Timeline (without heartbeats):
  ─────────────────────────────────────────────
  0s    User connects               → ONLINE
  10s   User's wifi dies            → still ONLINE (!!)
  1hr   ...                         → still ONLINE (!!)
  24hr  ...                         → STILL ONLINE?! 😱

  Timeline (with heartbeats + 60s TTL):
  ─────────────────────────────────────────────
  0s    User connects               → ONLINE (TTL: 60s)
  10s   User's wifi dies            → ONLINE (TTL: 50s)
  30s   No heartbeat received...    → ONLINE (TTL: 30s)
  60s   TTL expires!                → OFFLINE ✅
```

### How the heartbeat refreshes the TTL

```
  Client                    Server / Redis
  ──────                    ──────────────
       │                         │
       │── heartbeat ──────────▶│
       │                         │── SET presence:1 "online" EX 60
       │◀── heartbeat_ack ──────│   (TTL reset to 60s)
       │                         │
       │   ...30 seconds...      │
       │                         │   (TTL: 30s remaining)
       │── heartbeat ──────────▶│
       │                         │── SET presence:1 "online" EX 60
       │◀── heartbeat_ack ──────│   (TTL reset to 60s again!)
       │                         │
```

In [ ]:
# ── Watch the heartbeat refresh the TTL ──

alice_ws = ws_connect(WS_URL)
alice_ws.send(json.dumps({"type": "connect", "user_id": 1}))
json.loads(alice_ws.recv())  # connected ack

print("💓 Heartbeat Demo")
print("═" * 50)

# Check initial TTL
ttl = r.ttl("presence:1")
print(f"\n⏱️  After connect: TTL = {ttl}s")

# Wait 5 seconds, watch TTL decrease
print("\n⏳ Waiting 5 seconds (no heartbeat)...")
time.sleep(5)
ttl = r.ttl("presence:1")
print(f"⏱️  TTL is now: {ttl}s (decreased!)")

# Send a heartbeat
alice_ws.send(json.dumps({"type": "heartbeat"}))
hb_resp = json.loads(alice_ws.recv())
print(f"\n💓 Sent heartbeat → {hb_resp}")

# Check TTL again — it should be back to ~60
ttl = r.ttl("presence:1")
print(f"⏱️  TTL after heartbeat: {ttl}s (refreshed!)")

# Another wait + heartbeat cycle
print("\n⏳ Waiting 3 more seconds...")
time.sleep(3)
ttl = r.ttl("presence:1")
print(f"⏱️  TTL is now: {ttl}s")

alice_ws.send(json.dumps({"type": "heartbeat"}))
json.loads(alice_ws.recv())
ttl = r.ttl("presence:1")
print(f"💓 Heartbeat → TTL reset to: {ttl}s")

alice_ws.close()
print("\n🧹 Done! Connection closed.")

In [ ]:
# ── Scaling math: How many heartbeats per second? ──
#
# Let's think about scale like a real system designer.

print("📊 Heartbeat Scaling Analysis")
print("═" * 50)

# WhatsApp has ~2 billion users, ~200M are online at peak
online_users = 200_000_000   # 200 million
heartbeat_interval = 30      # seconds (typical)

pings_per_second = online_users / heartbeat_interval

print(f"\n📱 Online users (peak):     {online_users:>15,}")
print(f"⏱️  Heartbeat interval:      {heartbeat_interval:>15} seconds")
print(f"💓 Heartbeats per second:    {pings_per_second:>15,.0f}")
print(f"")
print(f"   That's {pings_per_second/1_000_000:.1f} MILLION pings per second! 🤯")
print(f"")
print(f"📦 Each heartbeat is tiny (~50 bytes), so bandwidth is:")

bytes_per_second = pings_per_second * 50
mb_per_second = bytes_per_second / (1024 * 1024)
gb_per_second = mb_per_second / 1024

print(f"   {mb_per_second:,.0f} MB/s = {gb_per_second:.1f} GB/s")
print(f"")
print(f"💡 How WhatsApp handles this:")
print(f"   • Heartbeats go to the edge server (not a central DB)")
print(f"   • Redis SET with EX (TTL) is O(1) — extremely fast")
print(f"   • Presence is sharded across many Redis instances")
print(f"   • Each server handles its own connected users' presence")

---

# 📈 Section 7: Scaling Read Receipts

Read receipts seem simple for 1:1 chats. But what about groups?

### The Problem: Group Read Receipts Explode

Imagine a group with 100 members. Alice sends a message.

```
  Alice sends 1 message to a group of 100:

  📩 99 inbox rows created (one per recipient)
  ✓✓ 99 ACK events when each person receives it
  🔵 99 read events when each person reads it

  Each read event notifies Alice:
  → 99 Redis PUBLISH commands
  → 99 WebSocket messages to Alice

  Total notifications for ONE message: 99 + 99 = 198 events!
```

Now imagine the group is active and people send 50 messages/hour:

```
  50 messages × 198 events = 9,900 events per hour
  That's 2.75 events per SECOND... for one group! 😰
```

### Real-World Strategies

#### 1. 📦 Batching

Instead of sending a notification for each read, batch them:

```
  WITHOUT batching:                WITH batching (every 5 seconds):
  ─────────────────                ──────────────────────────────────
  Bob read msg 42                  {"type": "read_batch",
  Charlie read msg 42              "chat_id": 3,
  Diana read msg 42                "reads": [
  Eve read msg 42                    {"msg": 42, "readers": [2,3,4,5]},
  Bob read msg 43                    {"msg": 43, "readers": [2]}
  → 5 separate events              ]}
                                   → 1 batched event
```

#### 2. 🔇 Skip Muted Chats

If Alice muted the group, don't bother sending her read receipts in
real time. Sync them lazily when she opens the chat.

#### 3. 👥 Large Group Limits

WhatsApp disables read receipts for groups with more than ~100 members.
Only delivery receipts (✓✓) are shown, not read receipts (🔵✓✓).

#### 4. 🗄️ Server-Side Aggregation

Instead of "Bob read message 42", store a counter:

```sql
  -- Instead of checking each reader:
  SELECT COUNT(*) FROM inbox WHERE message_id = 42 AND status = 'read';
  -- Result: 42 of 99 members have read it
```

The client just needs the count, not each individual event.

---

## 🧹 Cleanup

Let's clean up the data we created during this notebook so the next one
starts fresh.

In [ ]:
# Clean up messages and inbox rows we created during this notebook.
# We keep the original seed data intact.

# Remove inbox entries for messages beyond the seed data (id > 8)
execute("DELETE FROM inbox WHERE message_id > 8")
# Remove messages beyond the seed data
execute("DELETE FROM messages WHERE id > 8")
# Reset sequence counter for chat 1 back to seed value
execute("UPDATE chat_sequences SET last_sequence = 3 WHERE chat_id = 1")

# Clean up Redis presence keys
for uid in range(1, 6):
    r.delete(f"presence:{uid}")
    r.delete(f"last_seen:{uid}")

print("✅ Cleanup complete!")
print("   • Removed extra messages and inbox entries")
print("   • Reset sequence counters")
print("   • Cleared Redis presence keys")

---

# 🎓 Summary

In this notebook we explored two critical features of messaging apps:

## Key Takeaways

### ✅ Read Receipts

| Concept | What We Learned |
|---------|-----------------|
| **Inbox table** | One row per recipient per message, tracks `pending → delivered → read` |
| **ACK (✓✓)** | Client confirms it received the message → `status = 'delivered'` |
| **Read (🔵✓✓)** | Client confirms user opened the chat → `status = 'read'` |
| **Notification** | Read receipts flow back to the sender via Redis pub/sub |

### 🟢 Presence

| Concept | What We Learned |
|---------|-----------------|
| **Redis TTL** | `presence:{user_id}` key with 60s TTL = online |
| **Heartbeat** | Client pings every ~30s to refresh the TTL |
| **Last seen** | `last_seen:{user_id}` stores the disconnect timestamp |
| **Safety net** | If client crashes, TTL expires → auto-offline |

### 📈 Scaling Insights

| Challenge | Solution |
|-----------|----------|
| 200M heartbeats/30s | Edge servers + Redis sharding |
| Group read receipt explosion | Batching + large group limits |
| Muted chats | Lazy sync instead of real-time |

```
  What we built:

  ┌──────────┐     ┌──────────┐     ┌──────────────┐
  │  Client   │────▶│  Server   │────▶│  PostgreSQL   │
  │ (phone)   │◀────│ (WS+Redis)│◀────│  (inbox tbl)  │
  └──────────┘     └──────────┘     └──────────────┘
       │                │
       │  heartbeat     │  SET presence:N EX 60
       │  ──────────▶   │  ──────────▶  Redis
       │                │
       │  read receipt   │  PUBLISH user:N
       │  ◀──────────   │  ◀──────────  Redis
```

---

## ⏭️ Next Up: Notebook 3 — Group Messaging

In the next notebook we'll tackle **group chats** — how to fan out messages
to many participants efficiently, partitioning strategies, and admin controls.

See you there! 🚀